# 03 — Predict County
**MyVoterWisdom · [github.com/sysWisdom/myvoterwisdom](https://github.com/sysWisdom/myvoterwisdom)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sysWisdom/myvoterwisdom/blob/main/notebooks/03_predict_county.ipynb)

> Non-partisan educational tool. See [DISCLAIMER.md](../DISCLAIMER.md) before use.

This notebook runs the full `main_vote2028.py` prediction logic interactively.
Pick a county and state, then step through each model to see the classification results.

## Step 1 — Environment Setup

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('/content/myvoterwisdom'):
        os.system('git clone https://github.com/sysWisdom/myvoterwisdom.git /content/myvoterwisdom')
    REPO_ROOT = '/content/myvoterwisdom'
    os.system('pip install -q imbalanced-learn')
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

sys.path.insert(0, REPO_ROOT)
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'voting_pres_data.csv')
print(f"REPO_ROOT : {REPO_ROOT}")

## Step 2 — Select County & State

Edit the two variables below, then run all cells.

In [ ]:
import pandas as pd

# ── EDIT THESE TWO VALUES ──────────────────────────────────────────────────────
COUNTY_NAME = "Orange County"   # e.g. "Orange County", "Fulton County", "Clark County"
STATE_NAME  = "CA"              # Two-letter abbreviation, e.g. "CA", "GA", "NV"
# ──────────────────────────────────────────────────────────────────────────────

# Show available county / state combinations in the dataset
df_all = pd.read_csv(DATA_PATH)
print("Available counties in dataset:")
display(df_all.groupby('State')['County'].unique().apply(list).reset_index().rename(columns={'County': 'Counties'}))

## Step 3 — Run Prediction Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import numpy as np
import json
from preprocess import load_data, add_filter_columns, compare_votes_and_ballots, update_wisdom

def laplace_law_of_succession(wins, total):
    return (wins + 1) / (total + 2)

# ── Load & filter ─────────────────────────────────────────────────────────────
df = load_data(DATA_PATH)
county_data = df[(df['County'] == COUNTY_NAME) & (df['State'] == STATE_NAME)].copy()

if county_data.empty:
    print(f"❌ No data found for {COUNTY_NAME}, {STATE_NAME}. Check spelling above.")
else:
    print(f"✅ Found {len(county_data)} records for {COUNTY_NAME}, {STATE_NAME}")
    county_data = add_filter_columns(county_data)
    county_data = compare_votes_and_ballots(county_data)
    county_data = update_wisdom(county_data)

    # Feature engineering
    county_data['Democratic Vote Share'] = county_data['Democratic Votes'] / county_data['Total Voted']
    county_data['Republican Vote Share'] = county_data['Republican Votes'] / county_data['Total Voted']
    county_data['Turnout'] = county_data['Total Ballots Cast'] / county_data['Total Registered Voters']
    county_data['Democratic Wins'] = np.where(
        county_data['Democratic Vote Share'] > county_data['Republican Vote Share'], 1, 0
    )

    display(county_data[['Election Year', 'Democratic Vote Share', 'Republican Vote Share',
                          'Turnout', 'Democratic Wins']].round(3))

## Step 4 — Train Models & Display Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

if not county_data.empty:
    X = county_data[['Democratic Vote Share', 'Republican Vote Share', 'Turnout']]
    y = county_data['Democratic Wins']
    class_counts = y.value_counts()

    # ── Single-class guard ───────────────────────────────────────────────────
    if len(class_counts) < 2:
        likely = "Democratic" if y.iloc[0] == 1 else "Republican"
        print(f"ℹ️  Only one class in data. Using Laplace Law of Succession.")
        d_prob = laplace_law_of_succession(class_counts.get(1, 0), len(y)) * 100
        r_prob = laplace_law_of_succession(class_counts.get(0, 0), len(y)) * 100
        print(f"   Democratic probability : {d_prob:.1f}%")
        print(f"   Republican probability : {r_prob:.1f}%")
        print(f"   Likely winner          : {likely}")
    else:
        # Apply SMOTE if needed
        if class_counts.min() > 1:
            smote = SMOTE(random_state=42, k_neighbors=min(5, class_counts.min() - 1))
            X, y = smote.fit_resample(X, y)
            print(f"SMOTE applied. New class distribution: {dict(y.value_counts())}")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        models = {
            'Random Forest':      RandomForestClassifier(n_estimators=100, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'SVM':                SVC(),
            'Gradient Boosting':  GradientBoostingClassifier(),
        }

        results_summary = []
        for name, model in models.items():
            try:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                acc = report.get('accuracy', 0)
                results_summary.append({'Model': name, 'Accuracy': round(acc, 3)})
                print(f"\n── {name} ──")
                print(classification_report(y_test, y_pred, zero_division=0))
            except Exception as e:
                print(f"⚠️  {name} failed: {e}")

        # Summary bar chart
        if results_summary:
            summary_df = pd.DataFrame(results_summary).sort_values('Accuracy', ascending=True)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.barh(summary_df['Model'], summary_df['Accuracy'],
                    color=sns.color_palette('muted')[0], edgecolor='white')
            ax.set_xlim(0, 1)
            ax.set_xlabel('Accuracy')
            ax.set_title(f'Model Accuracy — {COUNTY_NAME}, {STATE_NAME}')
            for p in ax.patches:
                ax.annotate(f'{p.get_width():.2%}',
                            (p.get_width(), p.get_y() + p.get_height() / 2),
                            ha='left', va='center', fontsize=10, fontweight='bold')
            plt.tight_layout()
            plt.show()